# PhageScope Pipeline

Basic packages 

The PhageScope data are downloaded via snakemake and reports generated on up to 50k randomly selected datapoint. 




## File list



In [2]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

data_dir = "../../data"


### Protein Metadata

Temporary creation of big lists for later use. 

In [3]:
# Accumulators
chunk_num = 0
total_rows = 0
phage_ids = []
protein_ids = []
source = [] # Source of the protein, e.g. DDBJ, CHVD, etc. match the directory names for the fasta files


for chunk in pd.read_csv(os.path.join(data_dir, "merged/merged_annotated_proteins_metadata.csv"), chunksize=10000):
    total_rows += len(chunk)
    phage_ids.extend(chunk['Phage_ID'].tolist())
    protein_ids.extend(chunk['Protein_ID'].tolist())
    source.extend(chunk['Phage_source'].tolist())


    # ------------------------------
    chunk_num += 1
    if chunk_num >= 55:  # Limit to 5 chunks for demonstration
        break

In [4]:
print(pd.Series(source).unique())

['RefSeq' 'Genbank']


In [5]:

#transcription_terminator_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_transcription_terminator_metadata.csv"))
#phage_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_metadata.csv"))
#phage_protein_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_protein_metadata.csv"))

## Merging protein and phage Fasta

Some sources have one fasta file per protein, other have one fasta file containing all proteins. 

In total, there are more than 23 milion fasta files (just protein !) Some sources have one file containing multiple sequences, while other sources have one fasta file per sequence. 

```
Unique sources: 13
RefSeq: 480642 FASTA files
IGVD: 1 FASTA files
PhagesDB: 351898 FASTA files
GPD: 7616044 FASTA files
GOV2: 1 FASTA files
CHVD: 1 FASTA files
MGV: 10517011 FASTA files
DDBJ: 17391 FASTA files
STV: 1 FASTA files
EMBL: 11095 FASTA files
TemPhD: 3465586 FASTA files
GVD: 746146 FASTA files
Genbank: 217128 FASTA files
Total fasta files found: 23422945
```

To obtain the sequence from the protein ID, we should be able to query the sequences at will and for this, having a coeherent database is vital. We will then merge the sequences in one file per source which should allow later to create a performant database, which is not possible with 23 million files. 

The structure is like this: 

```
protein_fasta/
  └── source1/
        ├── source1/
        │     ├── phageA/
        │     │     ├── file1.fasta
        │     │     └── file2.fasta
        │     └── phageB/
        │           └── file3.fasta
  └── source2/
        └── single.fasta

```

### Merging Fasta files per source

Using the script `merge_protein_fasta.py` with the snakemake rules `snakemake --cores 2 data/protein_fasta_merged/DDBJ.fasta` for each file, it will merge the files if necessary into data/protein_fasta_merged. 





Visualize with `snakemake --cores 4 --dag | dot -Tsvg > dag_all.svg`

More complete command with cache and logging: `snakemake --cache --printshellcmds --reason --timestamp --cores 8 `

## Activation du cache pour les règles lourdes

Créer les répertoires nécessaires: 

```
sudo mkdir -p /mnt/snakemake-cache
sudo chown $(whoami) /mnt/snakemake-cache

```

`export SNAKEMAKE_OUTPUT_CACHE=/mnt/snakemake-cache/`

Le caching est activé pour les règles de téléchargement. 

Afin d'exécuter les scripts dans le bon environnement, on export l'environement pixi avec `pixi workspace export conda-environment -e base envs/pixi_base_enf.yaml`

On exécute snakemake depuis pixi (afin d'être sûr que ça soit le bon snakemake) et on donne les arguments conda nécessaires (exemple avec merge_annotated_proteins_metadata_tsvs): 

`pixi run snakemake all --cache --use-conda --conda-frontend mamba --printshellcmds --cores 2`

Attention: certain crashes peuvent être dûs à une surcharge I/O en cas de multithreading trop important. 



# Phage Metadata Exploration

Explore the data for further pip

In [6]:
import pandas as pd
import numpy as np
import os


In [7]:
# Read data/merged/merged_phage_metadata.csv
phage_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_metadata.csv"))
annotated_proteins_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_annotated_proteins_metadata.csv"))
phage_anti_crispr_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_anti_crispr_metadata.csv"))
phage_transmembrane_protein_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_transmembrane_protein_metadata.csv"))
phage_trna_tmrna_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_trna_tmrna_metadata.csv"))
phage_virulent_factor_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_virulent_factor_metadata.csv"))
transcription_terminator_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_transcription_terminator_metadata.csv"))


# For each Phage_source, get a sample of 10 phages in a new dataframe
sampled_phages = phage_metadata.groupby('Phage_source').apply(lambda x: x.sample(n=10, random_state=42)).reset_index(drop=True)

/tmp/ipykernel_113770/2354234772.py:3: DtypeWarning: Columns (2,19) have mixed types. Specify dtype option on import or set low_memory=False.
  annotated_proteins_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_annotated_proteins_metadata.csv"))
/tmp/ipykernel_113770/2354234772.py:6: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  phage_trna_tmrna_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_trna_tmrna_metadata.csv"))
/tmp/ipykernel_113770/2354234772.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_phages = phage_metadata.groupby('Phage_source').apply(lambda x: x.sample(n=10, random_state=42)).r

In [8]:
# Filter to keep only phage id and phage source
tmp = sampled_phages[['Phage_ID', 'Phage_source']]

# Get only rows with source PhagesDB
tmp[tmp['Phage_source'] == 'PhagesDB']

,Phage_ID,Phage_source
100,Actinoplanes_phage_phiAsp2,PhagesDB
101,Arthrobacter_phage_Shoya,PhagesDB
102,Microbacterium_phage_LeeroyJenkins,PhagesDB
103,Mycobacterium_phage_Kboogie,PhagesDB
104,Microbacterium_phage_Haunter,PhagesDB
105,Mycobacterium_phage_JC27,PhagesDB
106,Mycobacterium_phage_Funston,PhagesDB
107,Arthrobacter_phage_King2,PhagesDB
108,Mycobacterium_phage_Pharsalus,PhagesDB
109,Mycobacterium_phage_Jobu08,PhagesDB


In [9]:
sampled_phages

,Phage_ID,Length,GC_content,Taxonomy,Completeness,Host,Lifestyle,Cluster,Subcluster,Phage_source
0,SRS143070_a1_ct78652,41502,51.240904,Siphoviridae,High-quality,Rhodanobacter,virulent,cluster_199050,subcluster_240157,CHVD
1,SRS047126_a1_ct151_vs1,43900,43.200456,Siphoviridae,High-quality,Aggregatibacter,virulent,cluster_183566,subcluster_221512,CHVD
2,SRS148363_a1_ct32938_vs01,15525,42.344605,Caudovirales,Low-quality,Streptococcus,virulent,cluster_404679,subcluster_488716,CHVD
3,SAMN10080906_a1_ct1149,6590,45.402124,Caudovirales,Not-determined,Clostridiales,virulent,cluster_68414,subcluster_83140,CHVD
4,SAMN04261392_a1_ct225484_vs1,27150,46.243094,Siphoviridae,Medium-quality,Lactobacillus,virulent,cluster_470454,subcluster_566398,CHVD
...,...,...,...,...,...,...,...,...,...,...
135,TemPhD_cluster_38093,31049,52.391381,Caudovirales,High-quality,Escherichia coli,temperate,cluster_192866,subcluster_243224,TemPhD
136,TemPhD_cluster_2719,15736,44.922471,Caudovirales,Low-quality,Salmonella enterica,temperate,cluster_276597,subcluster_347385,TemPhD
137,TemPhD_cluster_3706,15747,44.910142,NaN,Low-quality,Salmonella enterica,temperate,cluster_133614,subcluster_168021,TemPhD
138,TemPhD_cluster_60053,45244,50.563611,Caudovirales,High-quality,Serratia marcescens,temperate,cluster_186765,subcluster_235570,TemPhD


In [10]:
annotated_proteins_metadata

,Phage_ID,Protein_source,Function_prediction_source,Start,Stop,Strand,Protein_ID,Product,Protein_classification,Molecular_weight,Aromaticity,Instability_index,Isoelectric_point,Helix_fraction,Turn_fraction,Sheet_fraction,Reduced_coefficient,Oxidized_coefficient,Phage_source,Function_Prediction_source
0,NC_001330.1,RefSeq,RefSeq,759,2243,+,NP_039590.1,replication initiation protein,replication;,407.4640,0.250000,17.125000,8.750052,0.250000,0.500000,0.000000,0,0,RefSeq,NaN
1,NC_001330.1,RefSeq,RefSeq,1805,2158,+,NP_039591.1,internal scaffolding protein,assembly;,5650.1306,0.234043,17.321489,7.018615,0.297872,0.212766,0.170213,9970,9970,RefSeq,NaN
2,NC_001330.1,RefSeq,RefSeq,2158,2319,+,NP_039592.1,hypothetical protein,hypothetical;,6235.2870,0.037736,37.058491,6.816041,0.339623,0.132075,0.377358,0,125,RefSeq,NaN
3,NC_001330.1,RefSeq,RefSeq,2270,2476,+,NP_039593.1,DNA maturation protein,infection;,8295.3801,0.161765,32.102941,9.300419,0.338235,0.132353,0.279412,22460,22460,RefSeq,NaN
4,NC_001330.1,RefSeq,RefSeq,2503,2955,+,NP_039594.1,external scaffolding protein,assembly;,1183.4075,0.000000,15.510000,11.999968,0.200000,0.200000,0.300000,0,0,RefSeq,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43088577,biochar_6181,prodigal,NaN,7353,7544,-,biochar_6181_12,unknown,unsorted;,7031.8013,0.015873,51.022222,6.219629,0.174603,0.158730,0.365079,1490,1490,STV,-
43088578,biochar_6181,prodigal,NaN,7544,8029,-,biochar_6181_13,unknown,unsorted;,2441.7423,0.095238,63.938095,8.685390,0.238095,0.238095,0.190476,0,0,STV,-
43088579,biochar_6181,prodigal,NaN,8049,8561,-,biochar_6181_14,Terminase,packaging;,3325.8150,0.100000,28.400000,5.850915,0.366667,0.166667,0.433333,5500,5500,STV,eggNOG-mapper
43088580,biochar_6181,prodigal,NaN,8804,9616,-,biochar_6181_15,Terminase,packaging;,6677.5315,0.116667,43.138333,4.308873,0.300000,0.183333,0.333333,23490,23615,STV,eggNOG-mapper


In [11]:
phage_anti_crispr_metadata

,Phage_ID,Protein_ID,Source,Phage_source
0,NC_061433.1,YP_010299115.1,Alignment;Acranker,RefSeq
1,NC_005178.1,NP_938237.1,Alignment;Acranker,RefSeq
2,NC_024387.1,YP_009044826.1,Alignment;Acranker,RefSeq
3,NC_061436.1,YP_010299280.1,Alignment;Acranker,RefSeq
4,NC_041851.1,YP_009591718.1,Alignment;Acranker,RefSeq
...,...,...,...,...
299827,biochar_855,biochar_855_95,Acranker,STV
299828,biochar_4575,biochar_4575_23,Acranker,STV
299829,biochar_826,biochar_826_1,Acranker,STV
299830,biochar_2844,biochar_2844_23,Acranker,STV


In [12]:
phage_transmembrane_protein_metadata

,Phage_ID,Protein_ID,Length,PredictedTMHsNumber,ExpnumberofAAsinTMHs,Expnumberfirst60AAs,TotalprobofNin,POSSIBLENterm,Insidesource,Insidestart,Insideend,TMhelixsource,TMhelixstart,TMhelixend,Outsidesource,Outsidestart,Outsideend,Phage_source
0,NC_001330.1,NP_039595.1,75,1,22.46252,22.46221,0.44551,True,TMHMM2.0,33.0,75.0,TMHMM2.0,10.0,32.0,TMHMM2.0,1.0,9.0,RefSeq
1,NC_001331.1,NP_039601.1,30,1,19.48607,19.48607,0.86987,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,30.0,RefSeq
2,NC_001331.1,NP_039602.1,83,1,23.01476,5.44076,0.03037,NaN,TMHMM2.0,80.0,83.0,TMHMM2.0,57.0,79.0,TMHMM2.0,1.0,56.0,RefSeq
3,NC_001331.1,NP_039603.1,82,2,43.70290,25.09221,0.99660,True,TMHMM2.0,80.0,82.0,TMHMM2.0,57.0,79.0,TMHMM2.0,43.0,56.0,RefSeq
4,NC_001331.1,NP_039604.1,437,2,37.26104,19.30140,0.90531,True,TMHMM2.0,436.0,437.0,TMHMM2.0,418.0,435.0,TMHMM2.0,27.0,417.0,RefSeq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2820227,biochar_6175,biochar_6175_6,52,2,36.49023,36.49023,0.34638,True,TMHMM2.0,24.0,29.0,TMHMM2.0,30.0,51.0,TMHMM2.0,52.0,52.0,STV
2820228,biochar_6175,biochar_6175_10,222,1,22.77331,22.75221,0.84202,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,222.0,STV
2820229,biochar_6175,biochar_6175_14,106,1,19.39243,19.39099,0.54127,True,TMHMM2.0,1.0,4.0,TMHMM2.0,5.0,24.0,TMHMM2.0,25.0,106.0,STV
2820230,biochar_6180,biochar_6180_1,812,1,21.67516,21.66885,0.97053,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,812.0,STV


In [13]:
phage_trna_tmrna_metadata

,t(m)RNA_ID,Source,t(m)RNA,Start,Stop,Strand,Length,Permuted,Sequence,Phage_ID,Phage_source
0,NC_001335.1-1-aragorn,aragorn,tRNA-Asn(gtt),4200,4272,forward,73,-,tgacgtgtagctcaatggcagagcgcccgactgttaatcgggtggt...,NC_001335.1,RefSeq
1,NC_001335.1-2-aragorn,aragorn,tRNA-Trp(cca),4350,4423,forward,74,-,gtacacgtagctcaattggtagagcagcggtctccaaagccgccgg...,NC_001335.1,RefSeq
2,NC_001335.1-3-aragorn,aragorn,tRNA-Gln(ctg),4479,4553,forward,75,-,tccccgttcgtctaatcggtaagacacccggctctggaccgggcaa...,NC_001335.1,RefSeq
3,NC_001835.1-1-aragorn,aragorn,tRNA-Trp(cca),27504,27576,forward,73,-,tgcgagcatagtataatggtaatgctacagattccaaacctgtaaa...,NC_001835.1,RefSeq
4,NC_001835.1-2-aragorn,aragorn,tRNA-?(Stop|Leu)(ta),27644,27718,forward,75,-,tcacgttatagcttagggaagagcagttctgtttaggcggaagcaa...,NC_001835.1,RefSeq
...,...,...,...,...,...,...,...,...,...,...,...
1298176,biochar_6137-5-tRNAscanSE,tRNAscanSE,tRNA-Trp(cca),7766,7837,reverse,72,-,tgagtgtgctaacggcaagctgctggcctccaaagccatgcgatct...,biochar_6137,STV
1298177,biochar_6137-6-tRNAscanSE,tRNAscanSE,tRNA-Ile2(cat),9150,9224,reverse,75,-,gggtccatagttcagtccggttagaacagtgacctcataagtcatt...,biochar_6137,STV
1298178,biochar_6150-1-tRNAscanSE,tRNAscanSE,tRNA-Leu(taa),6817,6901,reverse,85,-,tggtgcccaaggggggactcgaacccccacagattgctccgctgaa...,biochar_6150,STV
1298179,biochar_6150-2-tRNAscanSE,tRNAscanSE,tRNA-Ile2(cat),6653,6728,reverse,76,-,tggtgggtctgtaaggattcgaaccttcttctagcgattatgagtc...,biochar_6150,STV


In [14]:
# CHVD, IGVD and GOV2 seem to have problems with an invertion of Phage_ID 
phage_virulent_factor_metadata

,Protein_id,Aligned_Protein_in_VFDB,Phage_ID,Phage_source
0,YP_001449278.1,VFG001829(gb|WP_000752026),NC_004813.1,RefSeq
1,YP_009830126.1,VFG004787(gb|WP_000919350),NC_048634.1,RefSeq
2,YP_004286238.1,VFG000110(gb|WP_000979342),NC_015209.1,RefSeq
3,YP_009909342.1,VFG035045(gb|WP_001121226),NC_049944.1,RefSeq
4,NP_795658.1,VFG000951(gb|WP_011054794),NC_004588.1,RefSeq
...,...,...,...,...
23759,OTU_1211_46,VFG012939(gb|WP_000703656),OTU_1211,IGVD
23760,OTU_1796_17,VFG006815(gb|WP_010959001),OTU_1796,IGVD
23761,OTU_1208_17,VFG012939(gb|WP_000703656),OTU_1208,IGVD
23762,OTU_3600_1,VFG006815(gb|WP_010959001),OTU_3600,IGVD


In [15]:
transcription_terminator_metadata

,Phage_ID,Terminator,Start,Stop,Sense,Loc,Confidence,Phage_source
0,NC_001330.1,TERM_1,3037,3050,+,F,85,RefSeq
1,NC_001331.1,TERM_1,1171,1195,+,F,100,RefSeq
2,NC_001332.1,TERM_1,2279,2305,+,F,89,RefSeq
3,NC_001332.1,TERM_2,4925,4940,+,F,100,RefSeq
4,NC_001335.1,TERM_1,12352,12379,+,F,100,RefSeq
...,...,...,...,...,...,...,...,...
4640124,biochar_6173,TERM_1,3204,3231,+,F,84,STV
4640125,biochar_6175,TERM_1,5213,5251,+,F,93,STV
4640126,biochar_6175,TERM_2,8620,8650,+,F,93,STV
4640127,biochar_6181,TERM_1,905,953,+,F,100,STV
